# 03 - Enrich Crop + Weather (Silver)

1. Build `silver.province_mapping` (crop UPPERCASE -> weather canonical)
2. Map crop provinces; report unmapped / missing matches
3. Join weather on canonical province + year + quarter
4. Write `silver.crop_production_enriched`
5. Validate join coverage and uniqueness

## 1. Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

from configs.project_config import SILVER_TABLES, BUSINESS_KEY_COLS, VALID_QUARTERS

crop_clean = SILVER_TABLES['crop_production_clean']
weather_clean = SILVER_TABLES['weather_clean']
province_mapping_table = SILVER_TABLES['province_mapping']
enriched_table = SILVER_TABLES['crop_production_enriched']

print(f'Crop clean    : {crop_clean}')
print(f'Weather clean : {weather_clean}')
print(f'Province map  : {province_mapping_table}')
print(f'Enriched      : {enriched_table}')

## 2. Build province mapping

Crop uses UPPERCASE; weather uses Title Case with lowercase particles (`del`, `de`). Canonicalization keeps those particles lowercase.

In [0]:
crop_df = spark.table(crop_clean)
weather_df = spark.table(weather_clean)

crop_provinces = sorted([r.province for r in crop_df.select('province').distinct().collect()])
weather_provinces = sorted([r.province for r in weather_df.select('province').distinct().collect()])

print(f'Crop provinces    : {len(crop_provinces)}')
print(f'Weather provinces : {len(weather_provinces)}')

In [0]:
_LOWER_PARTICLES = {'del', 'de', 'la', 'las', 'los', 'y', 'van', 'von'}

def to_canonical(name: str) -> str:
    """UPPERCASE or mixed → weather-style Title Case (del/de stay lower)."""
    words = name.strip().split()
    out = []
    for i, w in enumerate(words):
        lw = w.lower()
        if i > 0 and lw in _LOWER_PARTICLES:
            out.append(lw)
        else:
            out.append(lw.capitalize())
    return ' '.join(out)

In [0]:
mapping_rows = [
    (src, to_canonical(src), 'standardize') for src in crop_provinces
]

mapping_schema = StructType([
    StructField('source_province', StringType(), False),
    StructField('canonical_province', StringType(), False),
    StructField('action', StringType(), False),
])
province_mapping = spark.createDataFrame(mapping_rows, schema=mapping_schema)

print('Province mapping:')
province_mapping.orderBy('source_province').show(60, truncate=False)

In [0]:
mapped_canonicals = {r.canonical_province for r in province_mapping.collect()}
weather_set = set(weather_provinces)

weather_only = sorted(weather_set - mapped_canonicals)
missing_in_weather = sorted(mapped_canonicals - weather_set)

print(f'Weather-only provinces (no crop rows): {weather_only}')
print(f'Mapped canonicals missing from weather: {missing_in_weather}')

if missing_in_weather:
    print('WARNING: fix to_canonical — these will not join to weather.')

## 3. Write province_mapping

In [0]:
(
    province_mapping.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(province_mapping_table)
)

print(f'Wrote: {province_mapping_table}')

## 4. Apply mapping + join weather

In [0]:
crop = spark.table(crop_clean)
weather = spark.table(weather_clean)
pmap = spark.table(province_mapping_table)

crop_mapped = (
    crop.alias('c')
    .join(pmap.alias('m'), F.col('c.province') == F.col('m.source_province'), 'left')
    .select('c.*', F.col('m.canonical_province').alias('canonical_province'))
)

weather_only_cols = [
    c for c in weather.columns
    if c not in {'province', 'year', 'quarter', '_ingested_at', '_source_file'}
]

weather_sel = weather.select(
    F.col('province').alias('w_province'),
    F.col('year').alias('w_year'),
    F.col('quarter').alias('w_quarter'),
    *[F.col(c).alias(f'w_{c}') for c in weather_only_cols]
)

enriched = (
    crop_mapped.alias('c')
    .join(
        weather_sel.alias('w'),
        (F.col('c.canonical_province') == F.col('w.w_province'))
        & (F.col('c.year') == F.col('w.w_year'))
        & (F.col('c.quarter') == F.col('w.w_quarter')),
        'left'
    )
    .drop('w_province', 'w_year', 'w_quarter')
)

## 5. Write enriched table

In [0]:
(
    enriched.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(enriched_table)
)

print(f'Wrote: {enriched_table}')

## 6. Data quality checks

In [0]:
result = spark.table(enriched_table)
n_rows = result.count()
n_keys = result.select(*BUSINESS_KEY_COLS).distinct().count()

print(f'Rows              : {n_rows}')
print(f'Distinct biz keys : {n_keys}')
print(f'Unique key?       : {n_rows == n_keys}')

checks = {
    'null province': result.filter(F.col('province').isNull()).count(),
    'null crop': result.filter(F.col('crop').isNull()).count(),
    'null year': result.filter(F.col('year').isNull()).count(),
    'null quarter': result.filter(F.col('quarter').isNull()).count(),
    'null production': result.filter(F.col('production').isNull()).count(),
    'production < 0': result.filter(F.col('production') < 0).count(),
    'invalid quarter': result.filter(~F.col('quarter').isin(VALID_QUARTERS)).count(),
    'unmapped province (canonical null)': result.filter(F.col('canonical_province').isNull()).count()
}

print('DQ checks (expect 0):')
for name, cnt in checks.items():
    print(f'  {name}: {cnt}')

if 'w_enso_phase' in result.columns:
    matched = result.filter(F.col('w_enso_phase').isNotNull()).count()
    print(f'Weather join match (w_enso_phase not null): {matched} / {n_rows}')

print()
print('Silver enrich complete.')
print('Next: Gold (03_gold/01_province_quarter_summary)')